In [1]:
import os
import pandas as pd
import yfinance as yf

DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

In [5]:
tickers = ["BTC-USD", "ETH-USD", "BNB-USD"]
start = "2020-01-01"
end = None # none = sampai data terbaru
interval = "1d" 

df = yf.download(
    tickers=tickers,
    start=start,
    end=end,
    interval=interval,
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

df.tail()

[*********************100%***********************]  3 of 3 completed


Ticker           BTC-USD                                            \
Price               Open          High           Low         Close   
Date                                                                 
2025-12-14  90296.437500  90498.109375  87634.937500  88175.179688   
2025-12-15  88171.078125  89983.921875  85304.078125  86419.781250   
2025-12-16  86424.406250  88170.093750  85381.687500  87843.984375   
2025-12-17  87847.617188  90264.570312  85316.265625  86143.757812   
2025-12-18  86196.789062  89279.960938  85871.429688  88191.429688   

Ticker                                     ETH-USD                            \
Price          Adj Close       Volume         Open         High          Low   
Date                                                                           
2025-12-14  88175.179688  50465972205  3116.743896  3128.622803  3034.692627   
2025-12-15  86419.781250  45559514323  3060.481445  3175.118164  2899.685791   
2025-12-16  87843.984375  41262178223  2964.379639  2978.921387  2890.011719   
2025-12-17  86143.757812  44243392914  2964.019531  3025.815430  2791.333984   
2025-12-18  88191.429688  42997706752  2832.820312  2993.499756  2822.574951   

Ticker                                                BNB-USD              \
Price             Close    Adj Close       Volume        Open        High   
Date                                                                        
2025-12-14  3060.594727  3060.594727  15619543350  897.435791  899.217651   
2025-12-15  2964.183105  2964.183105  28765976892  877.789246  894.205261   
2025-12-16  2964.180908  2964.180908  22709189818  858.545898  877.613098   
2025-12-17  2831.403564  2831.403564  26159825528  874.905518  875.110840   
2025-12-18  2949.294678  2949.294678  25117546496  842.925232  850.538025   

Ticker                                                      
Price              Low       Close   Adj Close      Volume  
Date                                                        
2025-12-14  874.129639  877.806885  877.806885  1640570319  
2025-12-15  843.767883  858.545105  858.545105  2222157806  
2025-12-16  851.367615  874.899048  874.899048  2379705373  
2025-12-17  834.851685  842.930176  842.930176  2968394043  
2025-12-18  830.797791  846.467468  846.467468  2813778688

In [6]:
if isinstance(df.columns, pd.MultiIndex):
    df_long = (
        df.stack(level=0)
        .rename_axis(["Date", "Ticker"])
        .reset_index()
        .sort_values(["Ticker", "Date"])
        .reset_index(drop=True)
    )
else:
    # fallback kalau cuma 1 ticker
    df_long = df.reset_index()
    df_long.insert(1, "Ticker", tickers[0])

df_long.head()


C:\Users\asusg\AppData\Local\Temp\ipykernel_2112\1594824804.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(level=0)


Price,Date,Ticker,Open,High,Low,Close,Adj Close,Volume
0,2020-01-01,BNB-USD,13.730962,13.873946,13.654942,13.689083,13.689083,172980718
1,2020-01-02,BNB-USD,13.698126,13.715548,12.989974,13.027011,13.027011,156376427
2,2020-01-03,BNB-USD,13.035329,13.763709,13.012638,13.660452,13.660452,173683857
3,2020-01-04,BNB-USD,13.667442,13.921914,13.560008,13.891512,13.891512,182230374
4,2020-01-05,BNB-USD,13.888340,14.410801,13.886547,14.111019,14.111019,202552703


In [7]:
print(df_long["Ticker"].unique())
print(df_long.groupby("Ticker")["Date"].agg(["min","max","count"]))
print("Missing Close:", df_long["Close"].isna().sum())

['BNB-USD' 'BTC-USD' 'ETH-USD']
               min        max  count
Ticker                              
BNB-USD 2020-01-01 2025-12-18   2179
BTC-USD 2020-01-01 2025-12-18   2179
ETH-USD 2020-01-01 2025-12-18   2179
Missing Close: 0


In [8]:
# Gabung 3 asset crypto menjadi 1 file csv
out_path = os.path.join(DATA_DIR, f"crypto_yahoo_{interval}_{start}_BTC_ETH_BNB.csv")
df_long.to_csv(out_path, index=False)
out_path

'../data/raw\\crypto_yahoo_1d_2020-01-01_BTC_ETH_BNB.csv'

In [9]:
# Simpan per-asset
for t in df_long["Ticker"].unique():
    out = os.path.join(DATA_DIR, f"{t}_{interval}_{start}.csv")
    df_long[df_long["Ticker"] == t].to_csv(out, index=False)
    print("Saved:", out)

Saved: ../data/raw\BNB-USD_1d_2020-01-01.csv
Saved: ../data/raw\BTC-USD_1d_2020-01-01.csv
Saved: ../data/raw\ETH-USD_1d_2020-01-01.csv
